# EV3 robustness of the field recovery (reviewer question: does it generalise beyond one split?)
Re-evaluates few-shot recalibration, conformal coverage and cost-aware alerts under **many different field splits**:
1. **Random grouped splits** – 50 re-draws of the pool / calibration / test split (near-duplicate groups kept together).
2. **Sequence-blocked splits** – images are ordered by camera file number (IMG_xxxx) and cut into contiguous blocks,
   so test images are never neighbours in time or row of training images (50 re-draws).
3. **Mixed-session test** – AUROC only on blocks in which healthy and frost-damaged vines were photographed together,
   to check that the model recognises damage rather than the acquisition session.
4. **Original Roboflow split** of the dataset authors.
5. **Calibration-set size** – conformal coverage for 20 / 50 / 100 / 184 calibration images (500 draws each).
6. Optional: full fine-tuning (fixed 12 epochs) on one sequence-blocked split.
Inputs: `ev3-frost-grape` and the EV3 notebook output (`notebook4873e635e9`: `weapd_<model>.pt`, `ev3_split.csv`).
**GPU T4 x2, Internet On.** Download `ev3_generalisation_small.zip`.

In [ ]:
CFG = dict(BACKBONES=["swin_tiny_patch4_window7_224", "convnext_tiny"], INITS=["weapd", "imagenet"], N_REP=50, K=50, ALPHA=0.05,
           COST_R=[5, 10], BLOCK=30, C=0.5, R=256, S=224, DO_FT=True, FT_EPOCHS=12, FT_LR=1e-4, OUT="/kaggle/working/ev3_generalisation")
import os, re, glob, json, time, numpy as np, pandas as pd, torch, torch.nn.functional as F, timm
from PIL import Image, ImageFile
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
ImageFile.LOAD_TRUNCATED_IMAGES = True; os.makedirs(CFG["OUT"], exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"; print(DEV)
PRETRAINED = os.environ.get("NO_PRETRAINED") is None

In [ ]:
# 1. Data: EV3 labels, near-duplicate groups and the split of the paper (ev3_split.csv)
ann = glob.glob("/kaggle/input/**/_annotations.coco.json", recursive=True); assert ann
EV3_ROOT = os.path.dirname(os.path.dirname(ann[0]))
SPLIT = next(iter(glob.glob("/kaggle/input/**/ev3_split.csv", recursive=True)), None); assert SPLIT, "attach the EV3 notebook output"
CKPT = {b: next(iter(glob.glob(f"/kaggle/input/**/weapd_{b}.pt", recursive=True)), None) for b in CFG["BACKBONES"]}
assert all(CKPT.values()), CKPT
ev = pd.read_csv(SPLIT); ev["k"] = ev.file.str.split("/").str[-2:].str.join("/")
files = {os.path.join(s, f): os.path.join(EV3_ROOT, s, f) for s in ["train", "valid", "test"] for f in os.listdir(os.path.join(EV3_ROOT, s)) if not f.endswith(".json")}
ev["path"] = ev.k.map(files); assert ev.path.notna().all()
ev["folder"] = ev.k.str.split("/").str[0]
ev["num"] = ev.k.str.extract(r"IMG_(\d+)")[0].astype(float)
Y = ev.y.values; N = len(ev)

# sequence blocks: contiguous runs of BLOCK images ordered by file number; files without a number form one extra block
order = ev.num.fillna(1e9).sort_values(kind="stable").index.values
blk = np.empty(N, int); blk[order] = np.arange(N) // CFG["BLOCK"]; blk[ev.num.isna().values] = blk.max() + 1
# merge blocks that share a near-duplicate group (union-find) so that no near-duplicate crosses a block boundary
par = {b: b for b in np.unique(blk)}
def find(a):
    while par[a] != a: par[a] = par[par[a]]; a = par[a]
    return a
for g, idx in ev.groupby("group").groups.items():
    bs = np.unique(blk[list(idx)])
    for b in bs[1:]: par[find(b)] = find(bs[0])
ev["block"] = [find(b) for b in blk]
bstat = ev.groupby("block").y.agg(["mean", "count"])
mixed_blocks = set(bstat.index[(bstat["mean"] >= 0.2) & (bstat["mean"] <= 0.8)])
ev["mixed"] = ev.block.isin(mixed_blocks)
print("blocks:", ev.block.nunique(), "| mixed blocks:", len(mixed_blocks), "images in mixed blocks:", int(ev.mixed.sum()),
      "(frost share %.2f)" % ev[ev.mixed].y.mean())
ev[["k", "y", "num", "group", "block", "mixed", "folder", "split"]].to_csv(os.path.join(CFG["OUT"], "ev3_blocks.csv"), index=False)

In [ ]:
# 2. Frozen features (computed once; saved so that every analysis below can be repeated on a CPU)
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def load(paths):
    return torch.stack([torch.from_numpy(np.asarray(Image.open(p).convert("RGB").resize((CFG["R"],) * 2, Image.BICUBIC))).permute(2, 0, 1) for p in paths])
X_img = load(ev.path)
def center(x): o = (CFG["R"] - CFG["S"]) // 2; return x[:, :, o:o + CFG["S"], o:o + CFG["S"]]
def make(b, init, n_cls=2):
    if init == "weapd":
        m = timm.create_model(b, pretrained=False, num_classes=11); m.load_state_dict(torch.load(CKPT[b], map_location="cpu"))
        if n_cls: m.reset_classifier(n_cls)
    else:
        m = timm.create_model(b, pretrained=PRETRAINED, num_classes=n_cls or 1000)
    return m.to(DEV)
@torch.no_grad()
def feats(m):
    m.eval(); out = []
    for i in range(0, N, 64):
        x = ((center(X_img[i:i + 64]).float() / 255 - MEAN) / STD).to(DEV)
        with torch.autocast("cuda", enabled=DEV == "cuda"):
            f = (m.forward_head(m.forward_features(x), pre_logits=True) + m.forward_head(m.forward_features(torch.flip(x, [3])), pre_logits=True)) / 2
        out.append(f.float().cpu())
    return torch.cat(out).numpy()
FEATS = {}
for b in CFG["BACKBONES"]:
    for init in CFG["INITS"]:
        m = make(b, init, n_cls=0 if init == "weapd" else 0); FEATS[(b, init)] = feats(m); del m
        if DEV == "cuda": torch.cuda.empty_cache()
np.savez_compressed(os.path.join(CFG["OUT"], "ev3_features_fp16.npz"), **{f"{b}__{i}": v.astype(np.float16) for (b, i), v in FEATS.items()})
print({k: v.shape for k, v in FEATS.items()})

In [ ]:
# 3. One evaluation = probe on training images, conformal + cost threshold on calibration images, metrics on test images
def lac_q(s, a):
    n = len(s); k = int(np.ceil((n + 1) * (1 - a))); return np.sort(s)[min(k, n) - 1]
def evaluate(Xf, tr, ca, te, rng, k=None):
    if k:  # k labelled images per class drawn from the training indices
        tr = np.concatenate([rng.choice(tr[Y[tr] == c], min(k, (Y[tr] == c).sum()), replace=False) for c in (0, 1)])
    sc = StandardScaler().fit(Xf[tr]); clf = LogisticRegression(C=CFG["C"], max_iter=5000).fit(sc.transform(Xf[tr]), Y[tr])
    pc = clf.predict_proba(sc.transform(Xf[ca])); pt = clf.predict_proba(sc.transform(Xf[te]))
    out = dict(test_auroc=roc_auc_score(Y[te], pt[:, 1]), test_bal_acc=balanced_accuracy_score(Y[te], pt[:, 1] >= 0.5))
    mt = ev.mixed.values[te]
    if mt.sum() > 10 and len(np.unique(Y[te][mt])) == 2: out["mixed_block_auroc"] = roc_auc_score(Y[te][mt], pt[mt, 1])
    q = lac_q(1 - pc[np.arange(len(ca)), Y[ca]], CFG["ALPHA"]); S_ = pt >= 1 - q
    out.update(coverage=S_[np.arange(len(te)), Y[te]].mean(), singleton=(S_.sum(1) == 1).mean(), defer=(S_.sum(1) == 2).mean(),
               missed_frost=((~S_[:, 1]) & (Y[te] == 1)).sum() / (Y[te] == 1).sum())
    ths = np.linspace(0.01, 0.99, 99)
    for r in CFG["COST_R"]:
        cc = [r * ((pc[:, 1] < t) & (Y[ca] == 1)).sum() + ((pc[:, 1] >= t) & (Y[ca] == 0)).sum() for t in ths]; t = ths[int(np.argmin(cc))]
        cost = lambda th: r * ((pt[:, 1] < th) & (Y[te] == 1)).sum() + ((pt[:, 1] >= th) & (Y[te] == 0)).sum()
        out[f"cost_reduction_r{r}"] = 1 - cost(t) / max(cost(0.5), 1)
    return out
def splits_from_groups(groups, seed):
    fold = np.zeros(N, int)
    for f, (_, te) in enumerate(StratifiedGroupKFold(10, shuffle=True, random_state=seed).split(np.zeros(N), Y, groups)): fold[te] = f
    return np.where(fold >= 5)[0], np.where((fold >= 3) & (fold < 5))[0], np.where(fold < 3)[0]

In [ ]:
# 4. Schemes: paper split, random grouped re-splits, sequence-blocked re-splits, original Roboflow split
rows = []; rng = np.random.default_rng(0)
SCHEMES = [("paper_split", [(np.where(ev.split == "pool")[0], np.where(ev.split == "cal")[0], np.where(ev.split == "test")[0])])]
SCHEMES.append(("random_grouped", [splits_from_groups(ev.group.values, 100 + r) for r in range(CFG["N_REP"])]))
SCHEMES.append(("sequence_blocked", [splits_from_groups(ev.block.values, 200 + r) for r in range(CFG["N_REP"])]))
rf = (np.where(ev.folder == "train")[0], np.where(ev.folder == "valid")[0], np.where(ev.folder == "test")[0])
leak = len(set(ev.group.values[rf[2]]) & set(ev.group.values[np.r_[rf[0], rf[1]]]))
print("Roboflow split: test groups shared with train/valid:", leak)
SCHEMES.append(("roboflow_original", [rf]))
for name, sp in SCHEMES:
    for rep, (tr, ca, te) in enumerate(sp):
        for (b, init), Xf in FEATS.items():
            for k in [CFG["K"], None]:
                r_ = evaluate(Xf, tr, ca, te, rng, k)
                rows.append(dict(scheme=name, rep=rep, backbone=b, init=init, k=k or "pool", n_train=len(tr), n_cal=len(ca), n_test=len(te), **r_))
    print(name, "done")
R_ = pd.DataFrame(rows); R_.to_csv(os.path.join(CFG["OUT"], "gen_runs.csv"), index=False)
met = ["test_auroc", "test_bal_acc", "mixed_block_auroc", "coverage", "singleton", "defer", "missed_frost", "cost_reduction_r5", "cost_reduction_r10"]
q = lambda p: (lambda x: np.nanpercentile(x, p))
S_ = R_.groupby(["scheme", "k", "backbone", "init"])[met].agg(["mean", q(2.5), q(97.5)]).round(4)
S_.to_csv(os.path.join(CFG["OUT"], "gen_summary.csv")); print(S_.xs("pool", level="k")[["test_auroc", "coverage", "cost_reduction_r10"]])
# share of re-splits in which the 95% coverage target was met (finite-sample: expected ~50% at exactly 0.95)
cov = R_[R_.scheme != "paper_split"].groupby(["scheme", "k", "backbone", "init"]).coverage.agg(
    below_0_93=lambda c: (c < 0.93).mean(), mean="mean", min="min").round(3)
cov.to_csv(os.path.join(CFG["OUT"], "gen_coverage_check.csv")); print(cov)

In [ ]:
# 5. Calibration-set size: coverage distribution over 500 random calibration draws (paper split, probe on the whole pool)
pool, cal, test = SCHEMES[0][1][0]; crow = []
for (b, init), Xf in FEATS.items():
    sc = StandardScaler().fit(Xf[pool]); clf = LogisticRegression(C=CFG["C"], max_iter=5000).fit(sc.transform(Xf[pool]), Y[pool])
    pc_all = clf.predict_proba(sc.transform(Xf[cal])); pt = clf.predict_proba(sc.transform(Xf[test]))
    for n_cal in [20, 50, 100, len(cal)]:
        for d in range(500 if n_cal < len(cal) else 1):
            sel = rng.choice(len(cal), n_cal, replace=False) if n_cal < len(cal) else np.arange(len(cal))
            qq = lac_q(1 - pc_all[sel, Y[cal][sel]], CFG["ALPHA"]); S2 = pt >= 1 - qq
            crow.append(dict(backbone=b, init=init, n_cal=n_cal, draw=d, coverage=S2[np.arange(len(test)), Y[test]].mean(), defer=(S2.sum(1) == 2).mean()))
C_ = pd.DataFrame(crow); C_.to_csv(os.path.join(CFG["OUT"], "calsize_runs.csv"), index=False)
CS = C_.groupby(["backbone", "init", "n_cal"]).agg(cov_mean=("coverage", "mean"), cov_p5=("coverage", lambda c: np.percentile(c, 5)),
                                                    cov_below_0_90=("coverage", lambda c: (c < 0.90).mean()), defer_mean=("defer", "mean")).round(3)
CS.to_csv(os.path.join(CFG["OUT"], "calsize_summary.csv")); print(CS)

In [ ]:
# 6. Optional: full fine-tuning (fixed 12 epochs, no model selection) on the first sequence-blocked split
if CFG["DO_FT"]:
    tr, ca, te = SCHEMES[2][1][0]; Yt = torch.tensor(Y); ft = []
    def aug(x):
        i, j = np.random.randint(0, CFG["R"] - CFG["S"] + 1, 2); x = x[:, :, i:i + CFG["S"], j:j + CFG["S"]]
        return torch.flip(x, [3]) if np.random.rand() < 0.5 else x
    def norm(x): return ((x.float() / 255 - MEAN) / STD).to(DEV)
    @torch.no_grad()
    def prob(m, idx):
        m.eval(); o = []
        for i in range(0, len(idx), 64):
            x = norm(center(X_img[idx[i:i + 64]]))
            with torch.autocast("cuda", enabled=DEV == "cuda"): o.append(((m(x) + m(torch.flip(x, [3]))) / 2).float().softmax(1)[:, 1].cpu())
        return torch.cat(o).numpy()
    for b in CFG["BACKBONES"]:
        for init in CFG["INITS"]:
            for s in [0, 1, 2]:
                torch.manual_seed(s); np.random.seed(s); m = make(b, init, 2)
                opt = torch.optim.AdamW(m.parameters(), lr=CFG["FT_LR"], weight_decay=0.05)
                steps = CFG["FT_EPOCHS"] * int(np.ceil(len(tr) / 32)); sch = torch.optim.lr_scheduler.OneCycleLR(opt, CFG["FT_LR"], total_steps=steps)
                scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
                for ep in range(CFG["FT_EPOCHS"]):
                    m.train(); perm = np.random.permutation(tr)
                    for i in range(0, len(perm), 32):
                        bi = perm[i:i + 32]; opt.zero_grad()
                        with torch.autocast("cuda", enabled=DEV == "cuda"): loss = F.cross_entropy(m(norm(aug(X_img[bi]))), Yt[bi].to(DEV))
                        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
                p = prob(m, te); mt = ev.mixed.values[te]
                ft.append(dict(backbone=b, init=init, seed=s, test_auroc=roc_auc_score(Y[te], p), test_bal_acc=balanced_accuracy_score(Y[te], p >= 0.5),
                               mixed_block_auroc=roc_auc_score(Y[te][mt], p[mt]) if mt.sum() > 10 and len(np.unique(Y[te][mt])) == 2 else np.nan))
                print(ft[-1]); del m
                if DEV == "cuda": torch.cuda.empty_cache()
    pd.DataFrame(ft).to_csv(os.path.join(CFG["OUT"], "ft_sequence_blocked.csv"), index=False)

In [ ]:
import zipfile
zp = "/kaggle/working/ev3_generalisation_small.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(CFG["OUT"]): z.write(os.path.join(CFG["OUT"], f), f)
print(zp, "done")